In [0]:
json_str = """{
  "name": "John Doe",
  "age": 35,
  "address": {
    "city": "Anytown",
    "state": "CA"
  },
  "children": [
    { "name": "Owen", "age": 10 },
    { "name": "Eva", "age": 8 }
  ]
}"""

In [0]:
import json
from pyspark.sql.functions import schema_of_json, lit

# Convert to JSON string if json_str is a dict (handles both types)
json_string = json.dumps(json_str) if isinstance(json_str, dict) else json_str

# Infer the DDL-formatted schema from the JSON string
inferred_schema = (
    spark.range(1)
    .select(schema_of_json(lit(json_string)).alias("schema"))
    .collect()[0][0]
)

print("Inferred schema (DDL format):")
print(inferred_schema)

Inferred schema (DDL format):
STRUCT<address: STRUCT<city: STRING, state: STRING>, age: BIGINT, children: ARRAY<STRUCT<age: BIGINT, name: STRING>>, name: STRING>


In [0]:
%sql
SELECT schema_of_json('{"name":"John Doe","age":35,"address":{"city":"Anytown","state":"CA"},"children":[{"name":"Owen","age":10},{"name":"Eva","age":8}]}') AS inferred_schema

inferred_schema
"STRUCT, age: BIGINT, children: ARRAY>, name: STRING>"


In [0]:
from pyspark.sql.functions import from_json, col

# Sample DataFrame with multiple JSON records as strings
json_rows = [
    ('{"name":"John Doe","age":35,"address":{"city":"Anytown","state":"CA"},"children":[{"name":"Owen","age":10},{"name":"Eva","age":8}]}',),
    ('{"name":"Jane Smith","age":28,"address":{"city":"Springfield","state":"IL"},"children":[{"name":"Liam","age":5}]}',),
]
df = spark.createDataFrame(json_rows, ["json_col"])

# Use the schema inferred above to parse the JSON column
parsed_df = df.select(from_json(col("json_col"), inferred_schema).alias("data"))

# Flatten top-level fields into individual columns
display(parsed_df.select("data.*"))

address,age,children,name
"List(Anytown, CA)",35,"List(List(10, Owen), List(8, Eva))",John Doe
"List(Springfield, IL)",28,"List(List(5, Liam))",Jane Smith


In [0]:
from pyspark.sql.functions import explode_outer, col

flattened_df = (
    parsed_df
    .select(
        col("data.name").alias("name"),
        col("data.age").alias("age"),
        col("data.address.city").alias("city"),
        col("data.address.state").alias("state"),
        explode_outer(col("data.children")).alias("child")
    )
    .select(
        "name",
        "age",
        "city",
        "state",
        col("child.name").alias("child_name"),
        col("child.age").alias("child_age")
    )
)

display(flattened_df)

name,age,city,state,child_name,child_age
John Doe,35,Anytown,CA,Owen,10
John Doe,35,Anytown,CA,Eva,8
Jane Smith,28,Springfield,IL,Liam,5


Simpler example

In [0]:
json_sample = '{"name": "Alice", "age": 30, "city": "Chennai"}' 
schema = spark.range(1).select(schema_of_json(lit(json_sample))).collect()[0][0]

print(schema)

STRUCT<age: BIGINT, city: STRING, name: STRING>


In [0]:
from pyspark.sql.functions import from_json, col

df = spark.createDataFrame([(json_sample,)], ["json_col"])

parsed_df = df.withColumn("parsed", from_json(col("json_col"), schema)) 
parsed_df.select("parsed.name", "parsed.age", "parsed.city").show()

+-----+---+-------+
| name|age|   city|
+-----+---+-------+
|Alice| 30|Chennai|
+-----+---+-------+

